# Bootstrap Tail Overestimation Test

**Hypothesis:** The multinomial logistic regression overestimates tail category
probabilities (P(7), P(8), P(9+)), and these errors compound when aggregating
to bet-level probabilities like P(3+).

**Test:** Bootstrap resample the training data, refit the model N times, and compare
the bootstrap-averaged predictions against point estimates and actual outcomes.

**Success criteria (define BEFORE running):**
- Tail categories show higher coefficient of variation across bootstrap iterations
- Bootstrap-averaged aggregated probabilities have lower Brier scores than point estimates
- Overconfidence on extreme predictions is reduced

## Imports

In [5]:
%load_ext autoreload
%autoreload 2

In [6]:
import numpy as np
import pandas as pd
import sqlite3
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from collections import Counter
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## Model Setup

In [1]:
# =============================================================
# CONFIG — UPDATE THESE
# =============================================================

N_BOOTSTRAPS = 200
RANDOM_SEED = 42
TEST_SIZE = 0.2

# UPDATE: your feature columns
FEATURE_COLS = [
    'cornerRatio',
 'abv_interaction',
 'crn_interaction',
 'mvAvgThrees',
 'daysBetweenGames',
 'wide_interaction',
 'home',
 'crn_fgaMv',
 'abv_fgaMv',
 'threes_residualsAllowedMv',
 'past3ThrPtPrct',
 'mvGood3Rate'
]

TARGET_COL = 'threesMade'


In [2]:
# =============================================================
# LOAD DATA — UPDATE CONNECTION AND QUERY
# =============================================================
from nba import NBAbase, NBAetl, NBAdata, NBAmodels
etl = NBAetl.etl()
data = NBAdata.data()
# threes = NBAmodels.models('threes')

# td = data.threes_pipe(threes.data)

# td = data.clean_na(td)

# train = td[td.game_date.between('2022-10-01','2024-04-12')]
# test = td[td.game_date.between('2024-10-01','2025-04-12')]
# val = td[td.game_date.between('2025-10-01','2026-04-11')]

In [ ]:
# =============================================================
# TRAIN/TEST SPLIT AND POINT ESTIMATE MODEL
# =============================================================

y_train = train.threesMade
y_test = test.threesMade
y_val = val.threesMade

x_train = train[FEATURE_COLS]
x_test = test[FEATURE_COLS]
x_val = val[FEATURE_COLS]

# standardize
scaler = StandardScaler()
x_train_scaled = pd.DataFrame(
    scaler.fit_transform(x_train),
    columns=FEATURE_COLS,
    index=x_train.index
)
x_test_scaled = pd.DataFrame(
    scaler.transform(x_test),
    columns=FEATURE_COLS,
    index=x_test.index
)

# add constant for statsmodels
x_train_sm = sm.add_constant(x_train_scaled)
x_test_sm = sm.add_constant(x_test_scaled)

print('Train: {:,}, Test: {:,}'.format(len(x_train), len(x_test)))

In [ ]:
for d in td[(td.wide_interaction.isna()) & (td.game_date>'2022-10-01')].game_date.unique():
    etl.update_shots_allowed([d])

In [ ]:
# fit point estimate model
point_model = sm.MNLogit(y_train, x_train_sm)
point_results = point_model.fit(disp=0)

point_probs = point_results.predict(x_test_sm)
point_classes = np.array(sorted(y_train.unique()))

print('Point model fitted')
print('Classes: {}'.format(point_classes))
print('\nPrediction shape: {}'.format(point_probs.shape))
print('Row sums (should be 1.0): {:.6f} to {:.6f}'.format(
    point_probs.sum(axis=1).min(), point_probs.sum(axis=1).max()))

## Bootstrap Function

In [ ]:
def fit_bootstrap_model(X_train_sm, y_train, X_test_sm):
    """
    Single bootstrap iteration: resample training data,
    fit MNLogit, return predicted probabilities and classes.
    """
    n = len(X_train_sm)
    idx = np.random.choice(n, size=n, replace=True)
    
    X_boot = X_train_sm.iloc[idx].reset_index(drop=True)
    y_boot = y_train.iloc[idx].reset_index(drop=True)
    
    model = sm.MNLogit(y_boot, X_boot)
    results = model.fit(disp=0, maxiter=200)
    
    probs = results.predict(X_test_sm)
    classes = np.array(sorted(y_boot.unique()))
    
    return probs, classes

In [ ]:
# =============================================================
# RUN THE BOOTSTRAP
# =============================================================

np.random.seed(RANDOM_SEED)

all_probs = []
all_classes = []
failed = 0

for i in range(N_BOOTSTRAPS):
    try:
        probs, classes = fit_bootstrap_model(X_train_sm, y_train, X_test_sm)
        all_probs.append(probs)
        all_classes.append(classes)
    except Exception as e:
        failed += 1
        if failed <= 3:
            print('Iteration {} failed: {}'.format(i, str(e)))
    
    if (i + 1) % 50 == 0:
        print('Completed {}/{} (failed: {})'.format(i + 1, N_BOOTSTRAPS, failed))

print('\nDone: {} successful, {} failed'.format(len(all_probs), failed))

## Bootstrap Results Analysis

### Test 1: Category Drop Rate
How often does the bootstrap lose a category entirely? High drop rate = not enough data to support that category.

In [ ]:
missing = Counter()
for classes in all_classes:
    for c in point_classes:
        if c not in classes:
            missing[c] += 1

n_success = len(all_classes)

print('Category Drop Rate Across {} Bootstrap Iterations'.format(n_success))
print('=' * 50)
for c in sorted(point_classes):
    ct = missing.get(c, 0)
    print('Category {:>3}: dropped {} times ({:.1f}%)'.format(
        c, ct, ct / n_success * 100))

# flag categories for potential collapse
collapse_threshold = 0.05
flagged = [c for c in point_classes if missing.get(c, 0) / n_success > collapse_threshold]
if flagged:
    print('\n>> Categories to consider collapsing (>{:.0f}% drop rate): {}'.format(
        collapse_threshold * 100, flagged))
else:
    print('\n>> All categories stable across bootstrap iterations')

### Test 2: Per-Category Uncertainty
Coefficient of variation for each outcome category. Tail categories should show higher CoV.

In [ ]:
# align all bootstrap probs to the same class ordering
n_test = len(X_test_sm)
n_classes = len(point_classes)

aligned_probs = []
for probs, classes in zip(all_probs, all_classes):
    aligned = np.zeros((n_test, n_classes))
    for j, c in enumerate(point_classes):
        if c in classes:
            # probs from statsmodels is a DataFrame or array
            prob_arr = probs.values if hasattr(probs, 'values') else probs
            idx = np.where(classes == c)[0][0]
            aligned[:, j] = prob_arr[:, idx]
    aligned_probs.append(aligned)

stacked = np.array(aligned_probs)  # (n_boots, n_test, n_classes)
print('Stacked shape: {}'.format(stacked.shape))

In [ ]:
# compute per-category stats
category_stats = []
for j, c in enumerate(point_classes):
    mean_prob = stacked[:, :, j].mean()
    std_prob = stacked[:, :, j].std(axis=0).mean()
    cov = std_prob / mean_prob if mean_prob > 1e-6 else np.inf
    category_stats.append({
        'category': c,
        'mean_prob': mean_prob,
        'boot_std': std_prob,
        'cov': cov
    })

cat_df = pd.DataFrame(category_stats)
print(cat_df.to_string(index=False))

In [ ]:
# plot: CoV by category
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(cat_df['category'].astype(str), cat_df['cov'])
axes[0].set_xlabel('Outcome Category')
axes[0].set_ylabel('Coefficient of Variation')
axes[0].set_title('Bootstrap Uncertainty by Category\n(Higher = less stable)')

axes[1].bar(cat_df['category'].astype(str), cat_df['mean_prob'])
axes[1].set_xlabel('Outcome Category')
axes[1].set_ylabel('Mean Predicted Probability')
axes[1].set_title('Mean Probability by Category')

plt.tight_layout()
plt.show()

### Test 3: Aggregated Bet Probabilities — Point vs Bootstrap
Compare calibration at each betting threshold.

In [ ]:
point_prob_arr = point_probs.values if hasattr(point_probs, 'values') else point_probs
y_test_arr = y_test.values if hasattr(y_test, 'values') else y_test

agg_results = []

for threshold in AGGREGATION_THRESHOLDS:
    # point estimate: sum categories >= threshold
    point_mask = point_classes >= threshold
    point_agg = point_prob_arr[:, point_mask].sum(axis=1)
    
    # bootstrap: aggregate each iteration then average
    boot_aggs = []
    for boot_idx in range(stacked.shape[0]):
        agg = np.zeros(n_test)
        for j, c in enumerate(point_classes):
            if c >= threshold:
                agg += stacked[boot_idx, :, j]
        boot_aggs.append(agg)
    
    boot_aggs = np.array(boot_aggs)
    boot_mean = boot_aggs.mean(axis=0)
    boot_std = boot_aggs.std(axis=0)
    
    # actuals
    actual = (y_test_arr >= threshold).astype(float)
    
    # metrics
    agg_results.append({
        'threshold': '{}+'.format(threshold),
        'actual_rate': actual.mean(),
        'point_pred': point_agg.mean(),
        'boot_pred': boot_mean.mean(),
        'shrinkage': (point_agg - boot_mean).mean(),
        'brier_point': ((point_agg - actual) ** 2).mean(),
        'brier_boot': ((boot_mean - actual) ** 2).mean(),
        'boot_uncertainty': boot_std.mean()
    })

agg_df = pd.DataFrame(agg_results)
print(agg_df.to_string(index=False))

In [ ]:
# plot: calibration comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

x = range(len(agg_df))
labels = agg_df['threshold'].values

# actual vs predicted
axes[0].plot(x, agg_df['actual_rate'], 'ko-', label='Actual')
axes[0].plot(x, agg_df['point_pred'], 's--', label='Point Est')
axes[0].plot(x, agg_df['boot_pred'], '^--', label='Bootstrap')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels)
axes[0].set_title('Calibration: Predicted vs Actual')
axes[0].legend()

# brier scores
width = 0.35
axes[1].bar([i - width/2 for i in x], agg_df['brier_point'], width, label='Point')
axes[1].bar([i + width/2 for i in x], agg_df['brier_boot'], width, label='Bootstrap')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].set_title('Brier Score (lower = better)')
axes[1].legend()

# shrinkage
axes[2].bar(x, agg_df['shrinkage'])
axes[2].set_xticks(x)
axes[2].set_xticklabels(labels)
axes[2].set_title('Shrinkage (point - bootstrap)\nPositive = bootstrap pulled down')
axes[2].axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

### Test 4: Extreme Predictions
Focus on the most confident predictions — the ones you would actually bet on.

In [ ]:
# analyze the top N most confident predictions at your primary bet threshold
BET_THRESHOLD = 3  # P(3+)
TOP_N = 50

point_mask = point_classes >= BET_THRESHOLD
point_agg = point_prob_arr[:, point_mask].sum(axis=1)

# bootstrap aggregation
boot_aggs = []
for boot_idx in range(stacked.shape[0]):
    agg = np.zeros(n_test)
    for j, c in enumerate(point_classes):
        if c >= BET_THRESHOLD:
            agg += stacked[boot_idx, :, j]
    boot_aggs.append(agg)
boot_aggs = np.array(boot_aggs)
boot_mean = boot_aggs.mean(axis=0)
boot_std = boot_aggs.std(axis=0)

actual = (y_test_arr >= BET_THRESHOLD).astype(float)

# top N most confident
top_idx = np.argsort(point_agg)[-TOP_N:]

print('Top {} Most Confident P({}+) Predictions'.format(TOP_N, BET_THRESHOLD))
print('=' * 50)
print('Point estimate mean:   {:.4f}'.format(point_agg[top_idx].mean()))
print('Bootstrap mean:        {:.4f}'.format(boot_mean[top_idx].mean()))
print('Actual hit rate:       {:.4f}'.format(actual[top_idx].mean()))
print('Shrinkage:             {:.4f}'.format(
    (point_agg[top_idx] - boot_mean[top_idx]).mean()))
print('Brier (point):         {:.4f}'.format(
    ((point_agg[top_idx] - actual[top_idx]) ** 2).mean()))
print('Brier (bootstrap):     {:.4f}'.format(
    ((boot_mean[top_idx] - actual[top_idx]) ** 2).mean()))

overconf_point = point_agg[top_idx].mean() - actual[top_idx].mean()
overconf_boot = boot_mean[top_idx].mean() - actual[top_idx].mean()

print('\nOverconfidence (point):     {:.4f}'.format(overconf_point))
print('Overconfidence (bootstrap): {:.4f}'.format(overconf_boot))

if abs(overconf_boot) < abs(overconf_point):
    pct = (1 - abs(overconf_boot) / abs(overconf_point)) * 100
    print('\n>> Bootstrap REDUCES overconfidence by {:.1f}%'.format(pct))
else:
    print('\n>> Bootstrap does NOT reduce overconfidence here')

In [ ]:
# plot: distribution of bootstrap uncertainty for top predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# scatter: point vs bootstrap for top predictions
axes[0].scatter(point_agg[top_idx], boot_mean[top_idx], alpha=0.6)
min_val = min(point_agg[top_idx].min(), boot_mean[top_idx].min())
max_val = max(point_agg[top_idx].max(), boot_mean[top_idx].max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', label='y=x')
axes[0].set_xlabel('Point Estimate P({}+)'.format(BET_THRESHOLD))
axes[0].set_ylabel('Bootstrap Mean P({}+)'.format(BET_THRESHOLD))
axes[0].set_title('Top {} Predictions: Point vs Bootstrap'.format(TOP_N))
axes[0].legend()

# histogram: bootstrap uncertainty (std) for top vs all predictions
axes[1].hist(boot_std, bins=30, alpha=0.5, label='All predictions', density=True)
axes[1].hist(boot_std[top_idx], bins=15, alpha=0.5, label='Top {} predictions'.format(TOP_N), density=True)
axes[1].set_xlabel('Bootstrap Std Dev')
axes[1].set_ylabel('Density')
axes[1].set_title('Uncertainty Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

### Summary

In [ ]:
print('=' * 60)
print('INTERPRETATION GUIDE')
print('=' * 60)
print()
print('1. DROP RATE: Categories dropped >{:.0f}% of the time'.format(
    collapse_threshold * 100))
print('   should be collapsed. Flagged: {}'.format(
    flagged if flagged else 'None'))
print()
print('2. CoV BY CATEGORY: If tail categories have 2-3x higher CoV')
print('   than core categories, the tail instability hypothesis is confirmed.')
print('   Max CoV: {} (category {})'.format(
    round(cat_df['cov'].max(), 4),
    cat_df.loc[cat_df['cov'].idxmax(), 'category']))
print('   Min CoV: {} (category {})'.format(
    round(cat_df.loc[cat_df['mean_prob'] > 0.01, 'cov'].min(), 4),
    cat_df.loc[cat_df.loc[cat_df['mean_prob'] > 0.01, 'cov'].idxmin(), 'category']))
print()
print('3. BRIER SCORES: Bootstrap wins at these thresholds:')
wins = agg_df[agg_df['brier_boot'] < agg_df['brier_point']]['threshold'].tolist()
print('   {}'.format(wins if wins else 'None'))
print()
print('4. EXTREME PREDICTIONS: Bootstrap overconfidence = {:.4f}'.format(overconf_boot))
print('   vs point overconfidence = {:.4f}'.format(overconf_point))